In [ ]:
import os
import os.path as osp
import numpy as np
import torch
import smplx

In [ ]:
AMASS_SEQ_DIR = "/mnt/d/ClothSim/AMASS/CMU/09/"         # folder with 01_03_poses.npz
MODEL_FOLDER  = "/mnt/d/ClothSim/"                      # folder with SMPL models (not the .npz sequence!)
GENDER        = "female"                                  # "male" / "female" / "neutral"
FORCE_GENDER  = True
FRAME_IDX     = 0                                       # which frame in the sequence
OUT_OBJ       = AMASS_SEQ_DIR + f"body_{GENDER}_frame000.obj"     # where to write the OBJ

In [ ]:
def find_poses_file(seq_dir: str) -> str:
    for f in os.listdir(seq_dir):
        if f.endswith("_poses.npz"):
            return osp.join(seq_dir, f)
    raise FileNotFoundError(f"No '*_poses.npz' in {seq_dir}")


def load_amass_shape(seq_dir: str):
    poses_path = find_poses_file(seq_dir)
    data = np.load(poses_path)

    betas = data.get("betas", None)
    gender = data.get("gender", None)
    if FORCE_GENDER:
        gender = GENDER

    # If there is a separate shape.npz, prefer that
    shape_path = osp.join(seq_dir, "shape.npz")
    if osp.exists(shape_path):
        sdata = np.load(shape_path)
        if "betas" in sdata:
            betas = sdata["betas"]

    if betas is None:
        betas = np.zeros(16, dtype=np.float32)

    if gender is None:
        gender = GENDER
    else:
        gender = str(gender)

    return betas, gender


def export_amass_canonical_body_to_obj(
    seq_dir: str,
    model_folder: str,
    out_obj: str,
):
    betas, gender = load_amass_shape(seq_dir)

    # Build SMPL model
    model = smplx.create(
        model_path=model_folder,
        model_type="smpl",
        gender=gender,
        use_pca=False,
        batch_size=1,
    )

    betas = np.asarray(betas, dtype=np.float32).reshape(1, -1)
    betas = betas[:, : model.num_betas]

    # Canonical pose: all zeros
    body_pose = np.zeros((1, 69), dtype=np.float32)     # 23 joints * 3
    global_orient = np.zeros((1, 3), dtype=np.float32)
    transl = np.zeros((1, 3), dtype=np.float32)

    out = model(
        betas=torch.from_numpy(betas).float(),
        body_pose=torch.from_numpy(body_pose).float(),
        global_orient=torch.from_numpy(global_orient).float(),
        transl=torch.from_numpy(transl).float(),
        return_verts=True,
    )

    verts = out.vertices[0].detach().cpu().numpy()
    faces = model.faces

    os.makedirs(osp.dirname(out_obj), exist_ok=True)
    with open(out_obj, "w") as f:
        for v in verts:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for tri in faces:
            f.write(f"f {tri[0] + 1} {tri[1] + 1} {tri[2] + 1}\n")

    print("Wrote canonical body OBJ:", out_obj)
    print("verts:", verts.shape, "faces:", faces.shape)


if __name__ == "__main__":
    export_amass_canonical_body_to_obj(
        AMASS_SEQ_DIR,
        MODEL_FOLDER,
        OUT_OBJ,
    )
